In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/KB_FinalProject_Team6/인허가_분석결과/stores_prepared.csv")

In [ ]:
df.head()

In [ ]:
import pandas as pd

print(f"행: {df.shape[0]:,}개 / 컬럼: {df.shape[1]}개")

column_info = pd.DataFrame({
    "컬럼명": df.columns,
    "자료형": df.dtypes.astype(str).to_numpy(),
    "결측수": df.isna().sum().to_numpy(),
    "결측률(%)": (df.isna().mean() * 100).round(2).to_numpy(),
    "고유값수": df.nunique().to_numpy()
})

with pd.option_context("display.max_rows", None):
    display(column_info)

In [ ]:
# 1. 식별용: 모델 입력에서는 제외
id_cols = [
    "통합점포ID"
]

# 2. 입력 후보: 과거 기준일 당시 유효한 값인지 확인 필요
candidate_cols = [
    "대분류",
    "업종명",
    "카드업종",
    "업태구분명",
    "구군",
    "법정동",
    "행정동코드",
    "lon",
    "lat"
]

# 3. 기간·규모 변수 재구성 참고용
# 현재 집계값을 과거 시점의 입력으로 바로 사용하지 않음
history_cols = [
    "인허가일자_dt",
    "최초인허가일",
    "최근인허가일",
    "인허가수",
    "업종수"
]

# 4. 정답 생성 참고용: 모델 입력에서는 제외
label_source_cols = [
    "폐업일자_dt",
    "영업여부"
]

selected_cols = (
    id_cols + candidate_cols + history_cols + label_source_cols
)

# 기존 df는 보존
df_selected = df.loc[:, selected_cols].copy()

# 행정동코드는 크기를 비교하는 숫자가 아니라 범주 코드
df_selected["행정동코드"] = (
    df_selected["행정동코드"].astype("string")
)

print("선택본 크기:", df_selected.shape)
display(df_selected.head())

In [ ]:
df.shape

In [ ]:
import pandas as pd

# 역할별로 남길 컬럼 정의
column_groups = {
    "식별용": [
        "통합점포ID"
    ],

    "입력 후보": [
        "대분류",
        "업종명",
        "카드업종",
        "업태구분명",
        "구군",
        "법정동",
        "행정동코드",
        "lon",
        "lat"
    ],

    "기간·규모 계산 참고용": [
        "최초인허가일",
        "최근인허가일",
        "인허가수",
        "업종수"
    ],

    "정답 생성 참고용": [
        "폐업일자_dt",
        "영업여부"
    ],

    "추가 검토용": [
        "프랜차이즈유무",
        "프랜차이즈_구분",
        "프랜차이즈_브랜드",
        "업종목록"
    ]
}

keep_cols = [
    col
    for cols in column_groups.values()
    for col in cols
]

# 필요한 컬럼 존재 여부 확인
missing_cols = [col for col in keep_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"df에 없는 컬럼: {missing_cols}")

# 기존 df를 보존하면서 선택본 생성
df_selected = df.loc[:, keep_cols].copy()

# 행정동코드는 숫자의 크기가 아닌 범주를 나타냄
df_selected["행정동코드"] = (
    df_selected["행정동코드"].astype("string")
)

# 선택본에서 제외한 컬럼 전체 목록
drop_cols = [col for col in df.columns if col not in keep_cols]

# 역할표
column_roles = pd.DataFrame([
    {"컬럼명": col, "역할": role}
    for role, cols in column_groups.items()
    for col in cols
])

print(f"기존 df: {df.shape}")
print(f"선택본: {df_selected.shape}")
print(f"유지: {len(keep_cols)}개 / 제외: {len(drop_cols)}개")

display(column_roles)
display(pd.DataFrame({"제외한 컬럼": drop_cols}))
display(df_selected.head())

In [ ]:
industry_cols = ["대분류", "업종명", "카드업종", "업태구분명"]

# 네 컬럼 중 하나라도 복합으로 표시된 점포
mixed_mask = df[industry_cols].isin(
    ["복합분류", "복합업종"]
).any(axis=1)

print(f"복합으로 표시된 점포: {mixed_mask.sum():,}개")

display(
    df.loc[
        mixed_mask,
        ["통합점포ID", "인허가수"] + industry_cols + ["업종목록"]
    ].head(20)
)

In [ ]:
import json
import pandas as pd

# 1. JSON 문자열을 실제 리스트로 변환
industry_lists = df["업종목록"].map(json.loads)

# 비어 있거나 잘못된 목록 확인
valid = industry_lists.map(
    lambda items:
        isinstance(items, list)
        and len(items) > 0
        and all(
            isinstance(item, str) and bool(item.strip())
            for item in items
        )
)

if not valid.all():
    raise ValueError(
        f"업종목록 확인이 필요한 행: {df.index[~valid].tolist()[:10]}"
    )

# 업종명 공백 제거 및 목록 안의 중복 제거
industry_lists = industry_lists.map(
    lambda items: sorted({item.strip() for item in items})
)

# 2. 업종을 행 단위로 펼친 후 0/1 인코딩
# 임시 행 번호를 사용해 기존 인덱스와 관계없이 처리
expanded = industry_lists.reset_index(drop=True).explode()

industry_flags = pd.get_dummies(
    expanded,
    prefix="업종",
    prefix_sep="__",
    dtype="uint8"
)

# 같은 점포의 업종들을 다시 한 행으로 묶기
industry_flags = industry_flags.groupby(
    level=0, sort=False
).max()

industry_flags = industry_flags.reindex(
    range(len(df)), fill_value=0
)
industry_flags.index = df.index

# 3. 기존 컬럼명과 충돌하는지 확인
overlap = industry_flags.columns.intersection(df.columns)
if len(overlap) > 0:
    raise ValueError(f"이미 존재하는 업종 컬럼: {overlap.tolist()}")

# 기존 업종명·업종목록을 새 업종별 컬럼으로 대체
df_encoded = pd.concat(
    [
        df.drop(columns=["업종명", "업종목록"]),
        industry_flags
    ],
    axis=1
)

# 4. 각 점포에서 1의 개수 = 목록의 업종 개수인지 검증
assert (
    industry_flags.sum(axis=1).to_numpy()
    == industry_lists.map(len).to_numpy()
).all()

print("기존 df:", df.shape)
print("생성한 업종 컬럼 수:", industry_flags.shape[1])
print("변환본 df_encoded:", df_encoded.shape)

# 복합업종 점포의 원래 목록과 변환 결과 확인
sample_index = industry_lists[industry_lists.map(len) > 1].head(5).index

display(df.loc[sample_index, ["통합점포ID", "업종목록"]])

# 샘플에서 실제로 1이 있는 업종 컬럼만 표시
sample_flags = industry_flags.loc[sample_index]
display(sample_flags.loc[:, sample_flags.any(axis=0)])

In [ ]:
df.columns

In [ ]:
import pandas as pd

key = "통합점포ID"

# df_encoded에서 생성된 업종별 컬럼만 가져오기
industry_cols = [
    col for col in df_encoded.columns
    if col.startswith("업종__")
]

if not industry_cols:
    raise ValueError("업종별 0/1 컬럼이 없습니다.")

industry_data = df_encoded[[key] + industry_cols].copy()

# 같은 점포끼리 정확히 연결되는지 확인
for name, frame in [("df", df), ("industry_data", industry_data)]:
    if frame[key].isna().any():
        raise ValueError(f"{name}의 통합점포ID에 결측이 있습니다.")
    if frame[key].duplicated().any():
        raise ValueError(f"{name}의 통합점포ID에 중복이 있습니다.")

if not df[key].isin(industry_data[key]).all():
    raise ValueError("업종별 정보가 없는 점포가 있습니다.")

overlap = set(industry_cols).intersection(df.columns)
if overlap:
    raise ValueError(f"df에 이미 존재하는 업종 컬럼: {sorted(overlap)}")

# 원본 컬럼을 모두 유지하고 업종 정보만 추가
df_master = df.merge(
    industry_data,
    on=key,
    how="left",
    validate="one_to_one",
    sort=False
)

print("원본 df:", df.shape)
print("결합용 df_master:", df_master.shape)
display(df_master.head())

In [ ]:
feature_cols = [
    "구군",
    "법정동",
    "행정동코드",
    "lon",
    "lat"
] + industry_cols

print(f"현재 모델 입력 후보: {len(feature_cols)}개")

# 외부 데이터는 df_master에 결합
# 기준일·정답·추가 변수를 확정한 뒤 학습 입력을 선택
# X = df_master.loc[:, feature_cols].copy()

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

save_path = "/content/drive/MyDrive/df_master.csv"

df_master.to_csv(
    save_path,
    index=False,
    encoding="utf-8-sig"
)

print(f"저장 완료: {save_path}")

In [ ]:
import pandas as pd

results = []

for col in ["점포ID", "통합점포ID"]:
    ids = df_master[col].astype("string").str.strip()

    # 결측·공란은 중복과 별도로 집계
    valid = ids.notna() & ids.ne("")
    duplicate_mask = valid & ids.duplicated(keep=False)

    results.append({
        "컬럼": col,
        "전체 행 수": len(df_master),
        "결측·공란 행 수": int((~valid).sum()),
        "중복된 ID 종류 수": ids[duplicate_mask].nunique(),
        "중복에 해당하는 행 수": int(duplicate_mask.sum()),
        "모든 행에 ID가 있고 고유한가": (
            bool(valid.all()) and not ids[valid].duplicated().any()
        )
    })

    if duplicate_mask.any():
        print(f"{col} 중복 행:")
        display(df_master.loc[duplicate_mask].head(20))

display(pd.DataFrame(results))

In [ ]:
df_master = df_master.loc[:, [
    "통합점포ID",
    *[col for col in df_master.columns if col != "통합점포ID"]
]].copy()

display(df_master.head())

In [ ]:
# 점포ID만 삭제하고 나머지 컬럼은 유지
df_master = df_master.drop(
    columns=["점포ID"],
    errors="ignore"
)

display(df_master.head())

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

df_master.to_csv(
    "/content/drive/MyDrive/df_master.csv",
    index=False,
    encoding="utf-8-sig"
)

print("구글 드라이브에 저장 완료")